In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, quarter, month, dayofmonth, date_format, explode, sequence, to_date, lit

# 1. Initialize Spark Session
spark = (
    SparkSession.builder
    .appName("Build_Dim_Date")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [2]:
# 2. Generate a continuous sequence of dates (2016 through 2018)
# We create a dummy dataframe just to hold the sequence generator
df = spark.createDataFrame([(1,)], ["dummy"])
date_df = df.select(
    explode(
        sequence(to_date(lit("2016-01-01")), to_date(lit("2018-12-31")))
    ).alias("full_date")
)

In [3]:
# 3. Extract the required date parts using PySpark time functions
dim_date = date_df.select(
    col("full_date"),
    year(col("full_date")).alias("year"),
    quarter(col("full_date")).alias("quarter"),
    month(col("full_date")).alias("month"),
    dayofmonth(col("full_date")).alias("day")
)

In [4]:
# 4. Create the integer 'date_key' (Standard format: YYYYMMDD)
dim_date = dim_date.withColumn(
    "date_key", 
    date_format(col("full_date"), "yyyyMMdd").cast("int")
)

In [6]:
# 5. Reorder columns to match the Hive DDL exactly
dim_date = dim_date.select(
    "date_key", 
    "full_date", 
    "year", 
    "quarter", 
    "month", 
    "day"
)
# Show the first few rows to verify
dim_date.show(5)

+--------+----------+----+-------+-----+---+
|date_key| full_date|year|quarter|month|day|
+--------+----------+----+-------+-----+---+
|20160101|2016-01-01|2016|      1|    1|  1|
|20160102|2016-01-02|2016|      1|    1|  2|
|20160103|2016-01-03|2016|      1|    1|  3|
|20160104|2016-01-04|2016|      1|    1|  4|
|20160105|2016-01-05|2016|      1|    1|  5|
+--------+----------+----+-------+-----+---+
only showing top 5 rows



In [7]:
# 6. Save to HDFS
dim_date.write.mode("overwrite").parquet("/user/student/dim_data/dim_date")